Objetivo:
Analisar alguns dados de voos do Departamento de Estatísticas de Transporte dos Estados Unidos

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS Spark_Project_Guide;
CREATE SCHEMA IF NOT EXISTS Spark_Project_Guide.flight_data;
CREATE VOLUME IF NOT EXISTS Spark_Project_Guide.flight_data.csv;
USE CATALOG Spark_Project_Guide;

In [0]:
volume_path = '/Volumes/spark_project_guide/flight_data/csv/'

In [0]:
flightData2015 = spark.read\
    .option("inferSchema", "true")\
    .option("header", "true")\
    .csv(f'{volume_path}/2015-summary.csv')

In [0]:
flightData2015.take(3)

In [0]:
flightData2015.sort("count").explain()

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "5")
display(flightData2015.sort("count").take(2))

In [0]:
flightData2015.createOrReplaceTempView("flight_data_2015")

In [0]:
#Verificando plano de execução do DataFrame e do SQL.(Mesmo plano)
sqlWay = spark.sql("""
    SELECT DEST_COUNTRY_NAME, count(1)
    FROM flight_data_2015
    GROUP BY DEST_COUNTRY_NAME

""")

dfWay = flightData2015\
    .groupBy("DEST_COUNTRY_NAME")\
    .count()

sqlWay.explain()
dfWay.explain()


In [0]:
spark.sql("SELECT max(count) from flight_data_2015").take(1)


In [0]:
from pyspark.sql.functions import max

flightData2015.select(max("count")).take(1)

Encontrar os cinco principais países de destino nos dados (número máximo de voos de/para qualquer localidade)

In [0]:
# Em SQL
maxSql = spark.sql("""
          SELECT DEST_COUNTRY_NAME, sum(count) as destination_total
          FROM flight_data_2015
          GROUP BY DEST_COUNTRY_NAME
          ORDER BY sum(count) DESC
          """)

maxSql.show(5)

In [0]:
# Em DataFrame
from pyspark.sql.functions import desc

flightData2015\
    .groupBy("DEST_COUNTRY_NAME")\
    .sum("count")\
    .withColumnRenamed("sum(count)", "destination_total")\
    .sort(desc("destination_total"))\
    .limit(5)\
    .show(5)


In [0]:
# plano de execução
from pyspark.sql.functions import desc

flightData2015\
    .groupBy("DEST_COUNTRY_NAME")\
    .sum("count")\
    .withColumnRenamed("sum(count)", "destination_total")\
    .sort(desc("destination_total"))\
    .limit(5)\
    .explain()
